In [1]:
from HINGE.evaluation_synthetic import run_synthetic_evaluation_with_dropout

from HINGE.feature_mapping import prepare_expression_features

from sklearn.preprocessing import normalize
import pickle
from pathlib import Path

import matplotlib.pyplot as plt

In [2]:
SAVE_DIR = Path('./hinge_checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
n_gene_comps = 128

In [5]:
with open(SAVE_DIR / 'checkpoint_step1.pkl', "rb") as f:
    data_dict=pickle.load(f)

data_dict

{'RNA_panel1': {'s1': AnnData object with n_obs × n_vars = 6989 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's2': AnnData object with n_obs × n_vars = 8978 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's3': AnnData object with n_obs × n_vars = 449221 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's4': AnnData object with n_obs × n_vars = 7442 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's5': AnnData object with n_obs × n_vars = 16540 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'f

In [6]:
#select which sample to use for the mapping

uni_480_list = [data_dict['HE_panel1'][1]]  # only the first sample for demonstration
uni_5k_list = [data_dict['HE_panel2'][1]]  # only the first sample for demonstration
rna_480_list = [data_dict['RNA_panel1']['s2']]  # only the first sample for demonstration
rna_5k_list = [data_dict['RNA_panel2']['s8']]  # only the first sample for demonstration

In [7]:
uni_480_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [8]:

uni_480_normalized_list = []

for adata in uni_480_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_480_normalized_list.append(adata_norm)

In [9]:
uni_5k_normalized_list = []

for adata in uni_5k_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_5k_normalized_list.append(adata_norm)

In [10]:
uni_480_normalized_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [11]:
adata_panel1          = rna_480_list[0]
uni_panel1_normed     = uni_480_normalized_list[0]
adata_panel2          = rna_5k_list[0]
uni_panel2_normed     = uni_5k_normalized_list[0]

# run test

In [17]:
df_dropout, fakes, stats = run_synthetic_evaluation_with_dropout(
    adata_panel1          = adata_panel1,
    uni_panel1_normed     = uni_panel1_normed,
    adata_panel2          = adata_panel2,
    uni_panel2_normed     = uni_panel2_normed,
    alpha_list            = [0.25, 0.5, 0.75, 1.0, 1.25],
    n_seeds_per_alpha     = 2,
    MAX_DIST              = None,
    dropout_frac          = 0.10,
    dropout_mode          = "independent",   # or "symmetric"
    SAMPLE_NAME           = "s1",
    evaluate_hinge = True
)

panel1 shape: (8978, 5101)
panel2 shape: (8899, 480)
  Computing population-level difference statistics
  Panel1: 8978 cells  |  Panel2: 8899 cells

  Coordinate displacement field:
    median dx        = -0.68 um
    median dy        = -0.19 um
    median NN dist   = 22.87 um
    kept fraction    = 100.0%

  UNI (normalized space, 2048D):
    |delta_mean| median = 0.00264
    delta_std    median = 0.01713

  Expression (444 genes):
    |delta_mean| median = 0.0622
    delta_std    median = 0.1445
  Created s1_t: alpha=0.25  seed=0  xy_method=displacement_field  mean coord shift=12.96 um  n=8978
  Created s1_t: alpha=0.25  seed=1  xy_method=displacement_field  mean coord shift=12.97 um  n=8978
  Created s1_t: alpha=0.50  seed=2  xy_method=displacement_field  mean coord shift=21.37 um  n=8978
  Created s1_t: alpha=0.50  seed=3  xy_method=displacement_field  mean coord shift=21.29 um  n=8978
  Created s1_t: alpha=0.75  seed=4  xy_method=displacement_field  mean coord shift=30.71 um  n=89